In [3]:
from sound_simulation import *
from beamforming import *
from sensor_array import *
from scene import *
from util import *
from psd_estimation import *
from plot_utils import *
from cropac import *
from ml_util import *

import numpy as np
import matplotlib.pyplot as plt
import os

In [4]:
fig_width = 8

duration = 30
sr = SR
c = 344
N = WINDOW_LENGTH
step = WINDOW_STEP
window = WINDOW
fmin = FMIN
fmax = FMAX
VN, f, validf = NSTFT, VALID_FREQUENCIES, VALID_FREQUENCY_INDICES

k = 2 * np.pi * f / c

sensor_array = array_circle8()

noise_audios = [
    normalize_channels(prepare_audio("../samples/meas1_ambience.wav", duration, sr, False)),
    pink_diffuse_noise(duration * sr, sensor_array.positions, sr)]
noise_spl = [60, 60]

def generate_random_pos(mind, maxd):
    dist = mind + np.random.rand() * (maxd-mind)
    p = np.random.randn(3)
    p *= dist / np.linalg.norm(p)
    p[2] = np.maximum(p[2], 1.0)
    return p

drone_audios = ["../samples/neo_flight.wav", "../samples/mini3_flight.wav", "../samples/mavic4_flight.wav"]
drone_spl = [65, 58, 68]
drone_name = ["neo", "mini3", "mavic4"]

NN = len(noise_audios)
ND = len(drone_audios)
NP = 5

target_positions = [
    [generate_random_pos(5, 40) for _ in range(NP)],
    [generate_random_pos(5, 15) for _ in range(NP)],
    [generate_random_pos(5, 25) for _ in range(NP)]]

noise_positions = [generate_random_pos(1, 2) for _ in range(NP)]


def save_sample(path, audio, target_udir):
    pathbase = path
    idx = 0
    while os.path.isfile(f"{pathbase}_{idx}.wav"):
        idx += 1
    sf.write(f"{pathbase}_{idx}.wav", audio.T, sr)
    with open(f"{pathbase}_{idx}.json", "w", encoding="utf-8") as fil:
        json.dump(target_udir.tolist(), fil)

def max_norm_audio(audio):
    fact = 1 / np.max(np.abs(audio))
    return audio * fact

for ni in range(NN):

    for ip in range(NP):

        for di in range(ND):

            results = []

            target_position = target_positions[di][ip]
            scene = scene_stationary(duration, sr, target_position)
            target_gain = db2lin(drone_spl[di]-noise_spl[ni])
            target_sound = prepare_drone_audio(scene, sensor_array, drone_audios[di], duration, sr, target_gain)
            audio = noise_audios[ni] + target_sound
            target_udir = target_position / np.linalg.norm(target_position)

            save_sample(f"../dataset/synthetic_training/{drone_name[di]}", max_norm_audio(audio), target_udir)

for ni in range(NN):

    for ip in range(NP):

        results = []

        target_position = noise_positions[ip]
        target_udir = target_position / np.linalg.norm(target_position)

        save_sample(f"../dataset/synthetic_training/noise", max_norm_audio(noise_audios[ni]), target_udir)


In [6]:
fig_width = 8

duration = 3
sr = SR
c = 344
N = WINDOW_LENGTH
step = WINDOW_STEP
window = WINDOW
fmin = FMIN
fmax = FMAX
VN, f, validf = NSTFT, VALID_FREQUENCIES, VALID_FREQUENCY_INDICES

k = 2 * np.pi * f / c

sensor_array = array_circle8()

noise_audios = [
    normalize_channels(prepare_audio("../samples/meas1_ambience.wav", duration, sr, False))]
noise_spl = [60]

drone_audios = ["../samples/neo_flight.wav", "../samples/mini3_flight.wav", "../samples/mavic4_flight.wav"]
drone_spl = [65, 58, 68]
drone_name = ["neo", "mini3", "mavic4"]

NN = len(noise_audios)
ND = len(drone_audios)

target_distances = np.array(range(0, 151, 10))
target_positions = [np.array([i, 0, 10]) for i in target_distances]

NP = len(target_distances)


def save_sample2(path, audio, target_udir):
    pathbase = path
    sf.write(f"{pathbase}.wav", audio.T, sr)
    with open(f"{pathbase}.json", "w", encoding="utf-8") as fil:
        json.dump(target_udir.tolist(), fil)

def max_norm_audio(audio):
    fact = 1 / np.max(np.abs(audio))
    return audio * fact

for ni in range(NN):
    for ip in range(NP):
        for di in range(ND):

            results = []

            target_position = target_positions[ip]
            scene = scene_stationary(duration, sr, target_position)
            target_gain = db2lin(drone_spl[di]-noise_spl[ni])
            target_sound = prepare_drone_audio(scene, sensor_array, drone_audios[di], duration, sr, target_gain)
            audio = noise_audios[ni] + target_sound
            target_udir = target_position / np.linalg.norm(target_position)

            save_sample2(f"../dataset/synthetic_meas/{drone_name[di]}_{int(target_distances[ip])}", max_norm_audio(audio), target_udir)

